# NYC Mobility - Bronze Validation

## Why validate Bronze

Before moving to Silver, we confirm that Bronze received what we expected and that repeated ingestion did not duplicate batches. These checks report issues; they do not fix or remove rows.


## Green Taxi row count by source file

The recorded load evidence is March 44,208, April 44,238, and May 44,921. Run this consolidated check against the current Bronze table.


In [0]:
-- validate bronze row counts by source file

SELECT
    source_file,
    COUNT(*) AS bronze_row_count
FROM `ftw-week-08`.`01_bronze`.`green_taxi`
GROUP BY source_file
ORDER BY source_file;


## Total Bronze rows

The verified final Green Taxi Bronze total is **133,367 rows**.


In [0]:
SELECT COUNT(*) AS total_bronze_rows
FROM `ftw-week-08`.`01_bronze`.`green_taxi`;

## Ingestion log

The original evidence shows one successful record for each March, April, and May source file, with row counts matching the loaded batches.


In [0]:
SELECT
    source_identifier,
    batch_id,
    status,
    rows_loaded,
    ingested_at
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
WHERE source_name = 'green_taxi'
ORDER BY ingested_at;

This additional check makes duplicate log records visible per source identifier and batch.


In [0]:
-- validate one successful log record per monthly batch

SELECT
    source_identifier,
    batch_id,
    status,
    rows_loaded,
    COUNT(*) AS log_records
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
WHERE source_name = 'green_taxi'
GROUP BY
    source_identifier,
    batch_id,
    status,
    rows_loaded
ORDER BY source_identifier;


The original May-specific check also recorded one log row.


In [0]:
SELECT COUNT(*) AS may_log_rows
FROM `ftw-week-08`.`01_bronze`.`ingestion_log`
WHERE source_identifier = 'green_tripdata_2026-05.parquet';

## Idempotency evidence

We deliberately reran March and May in the Bronze load notebook. Both second runs inserted **0 rows**. March stayed at 44,208 rows, May did not create another batch, and the final total remained 133,367.


## Provenance

Every loaded Green Taxi row should carry its source file, batch ID, source system, and ingestion timestamp.


In [0]:
-- validate bronze provenance metadata

SELECT
    source_file,
    batch_id,
    source_system,
    COUNT(*) AS row_count,
    SUM(
        CASE
            WHEN source_file IS NULL
              OR batch_id IS NULL
              OR source_system IS NULL
              OR ingested_at IS NULL
            THEN 1
            ELSE 0
        END
    ) AS rows_with_missing_provenance
FROM `ftw-week-08`.`01_bronze`.`green_taxi`
GROUP BY
    source_file,
    batch_id,
    source_system
ORDER BY source_file;


## Source-to-Bronze reconciliation

This compares each verified landed Parquet count with its Bronze count. It does not modify either side.


In [0]:
-- reconcile each source file with bronze

WITH source_counts AS (
    SELECT
        'green_tripdata_2026-03.parquet' AS source_file,
        COUNT(*) AS source_row_count
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-03.parquet',
        format => 'parquet'
    )

    UNION ALL

    SELECT
        'green_tripdata_2026-04.parquet' AS source_file,
        COUNT(*) AS source_row_count
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-04.parquet',
        format => 'parquet'
    )

    UNION ALL

    SELECT
        'green_tripdata_2026-05.parquet' AS source_file,
        COUNT(*) AS source_row_count
    FROM read_files(
        '/Volumes/ftw-week-08/00_source_inspection/cloudfare-r2/groups/week-08/group-b/source/green_taxi/green_tripdata_2026-05.parquet',
        format => 'parquet'
    )
),

bronze_counts AS (
    SELECT
        source_file,
        COUNT(*) AS bronze_row_count
    FROM `ftw-week-08`.`01_bronze`.`green_taxi`
    GROUP BY source_file
)

SELECT
    source.source_file,
    source.source_row_count,
    COALESCE(bronze.bronze_row_count, 0) AS bronze_row_count,
    source.source_row_count = COALESCE(bronze.bronze_row_count, 0) AS counts_match
FROM source_counts AS source
LEFT JOIN bronze_counts AS bronze
    ON source.source_file = bronze.source_file
ORDER BY source.source_file;


## Remaining Bronze Work

- Taxi Zones Bronze ingestion: not yet implemented
- Weather Bronze ingestion: not yet implemented
- Traffic Advisory Bronze ingestion: bonus, not yet implemented
- Full Bronze quality gate: pending completion of all required Bronze sources

We stop here before Silver. The next step is to implement and validate the missing required Bronze sources without cleaning business values in Bronze.
